# Evaluate the Research Assistant

Run this notebook to check application behavior, measure retrieval quality, and review generated answers. The default run uses simulated providers and needs no API keys. Live checks are controlled by three independent switches.

From the repository root, install the dependencies and launch the notebook in the same Python environment.

```powershell
python -m pip install -r requirements.txt notebook ipykernel
jupyter notebook evaluationNotebook.ipynb
```

For live checks, put `GOOGLE_API_KEY` (or `GEMINI_API_KEY`) in a local `.env` file. Add `TAVILY_API_KEY` for the live smoke test and for web evidence in the optional answer review. Update `ENV_FILE` below if your file is elsewhere.

In [1]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
CASE_FILE = PROJECT_ROOT / "evaluation" / "cases.json"

if not CASE_FILE.is_file() or not (PROJECT_ROOT / "requirements.txt").is_file():
    raise RuntimeError("Open this notebook from the Research Assistant repository root.")

ENV_FILE = PROJECT_ROOT / ".env"
RUN_LIVE_SMOKE = False
RUN_LIVE_RETRIEVAL = False
RUN_LIVE_ANSWER_REVIEW = False

cases = json.loads(CASE_FILE.read_text(encoding="utf-8"))
cases_by_id = {case["id"]: case for case in cases}

## 1. Check application behavior

The offline test suite covers query preparation, bounded retries, citation validation, session isolation, and the API and CLI entry points.

In [2]:
test_run = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if test_run.returncode:
    print(test_run.stdout + test_run.stderr)
    raise RuntimeError("The offline test suite failed. See the test output above.")

summary = (test_run.stdout + test_run.stderr).strip().splitlines()
print("\n".join(summary[-3:]))

Ran 23 tests in 0.569s

OK


## 2. Inspect the labeled questions

The benchmark fixture has 24 cases across exact terms, paraphrases, follow-ups, and out-of-corpus questions. Each answerable case maps expected document IDs to graded relevance. Follow-up cases include prior conversation. Cases with an empty relevance map are counted separately when measuring how often the system returns evidence outside the corpus.

For comparisons, keep the labeled questions and corpus fixed across runs. [Benchmark fixture](evaluation/cases.json)

In [3]:
group_counts = Counter(case["group"] for case in cases)
print(f"{len(cases)} total cases")
for group, count in sorted(group_counts.items()):
    print(f"  {group.replace('_', ' ')}  {count}")

example = next(case for case in cases if case["group"] == "follow_up")
print("\nExample follow-up fixture")
print(json.dumps(example, ensure_ascii=False, indent=2))

24 total cases
  exact  9
  follow up  4
  out of corpus  3
  paraphrase  8

Example follow-up fixture
{
  "id": "follow-rag",
  "group": "follow_up",
  "topic": "Which tools can store the evidence it looks up?",
  "history": [
    "How does retrieval-augmented generation work?",
    "RAG retrieves supporting passages from a knowledge base and supplies them to the language model."
  ],
  "relevance": {
    "doc_002": 3
  }
}


## 3. Measure retrieval offline

The benchmark compares **raw-topic TF-IDF**, **first-round hybrid ranking**, and **final selected evidence** after reranking and any coverage retry.

- **Recall@5** measures the share of labeled relevant documents found in the top five.
- **nDCG@5** rewards placing the most relevant documents higher in the top five. Both ranking scores range from 0 to 1 and use the answerable cases.
- **Out-of-corpus hits** count how many out-of-corpus questions returned any evidence. Fewer hits indicate better abstention on this fixture.
- **Mean and p95 latency** summarize wall-clock retrieval time. The p95 describes the slower end of the run.

The offline numbers come from deterministic providers. Use this run to check that the harness and scoring work.

In [4]:
def run_benchmark(*args):
    process = subprocess.run(
        [sys.executable, "-m", "evaluation.retrieval_benchmark", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    try:
        report = json.loads(process.stdout)
    except json.JSONDecodeError as exc:
        detail = process.stderr.strip() or process.stdout.strip()
        raise RuntimeError(f"Benchmark exited with code {process.returncode}.\n{detail}") from exc

    # The benchmark still emits a report when the live gate fails.
    return report


def show_retrieval_report(report):
    print(f"{report['mode']}  {report['scored_cases']}/{report['cases']} cases scored")
    for stage, metrics in report["metrics"].items():
        print(f"\n{stage.replace('_', ' ')}")
        print(f"  Recall@5 {metrics['recall_at_5']:.3f}  nDCG@5 {metrics['ndcg_at_5']:.3f}")
        print(f"  Mean {metrics['mean_ms']:.2f} ms  p95 {metrics['p95_ms']:.2f} ms")
        print(f"  Out-of-corpus hits {metrics['out_of_corpus_with_results']}/{metrics['out_of_corpus_total']}")

    if "quality_gate_status" in report:
        print("\nQuality gate", report["quality_gate_status"])
        for field in ("provider_failures", "graph_fallbacks", "regressed_cases"):
            if report.get(field):
                print(field.replace("_", " "), report[field])


offline_report = run_benchmark()
show_retrieval_report(offline_report)

offline_simulation  21/24 cases scored

lexical
  Recall@5 0.619  nDCG@5 0.601
  Mean 0.36 ms  p95 0.48 ms
  Out-of-corpus hits 1/3

hybrid before rerank
  Recall@5 0.857  nDCG@5 0.794
  Mean 5.88 ms  p95 10.44 ms
  Out-of-corpus hits 3/3

full retrieval
  Recall@5 0.619  nDCG@5 0.618
  Mean 7.23 ms  p95 11.74 ms
  Out-of-corpus hits 1/3


## 4. Run optional live checks

Enable either switch in the setup cell and rerun the setup and the matching cell below. Live runs make API calls, and the 24-case run can take several minutes.

The **smoke test** checks a single real turn through Gemini, Tavily, and answer generation. The **live retrieval benchmark** uses Gemini and the bundled corpus, with web search disabled and answer generation stubbed. The retrieval gate passes when both ranking metrics meet or exceed TF-IDF and the run completes without fallbacks. A `provider_error` identifies a provider failure or fallback. A `ranking_regression` identifies a clean run that missed the baseline on at least one ranking metric.

In [5]:
if RUN_LIVE_SMOKE:
    smoke_report = run_benchmark("--smoke", "--env-file", str(ENV_FILE))
    print(json.dumps(smoke_report, indent=2))
else:
    print("Set RUN_LIVE_SMOKE = True to check Gemini and Tavily.")

Set RUN_LIVE_SMOKE = True to check Gemini and Tavily.


In [6]:
if RUN_LIVE_RETRIEVAL:
    live_report = run_benchmark("--live", "--env-file", str(ENV_FILE))
    show_retrieval_report(live_report)
else:
    print("Set RUN_LIVE_RETRIEVAL = True to score live retrieval.")

Set RUN_LIVE_RETRIEVAL = True to score live retrieval.


## 5. Check the deployed API

Set `PREVIEW_API_URL` to the FastAPI preview URL. The probe sends a first turn, a follow-up using the same thread, and a small batch on separate threads. Start with a fresh rate-limit window. The reported p95 is for this small sample. Review backend logs for provider fallbacks and use the next section to judge answer quality.

In [7]:
PREVIEW_API_URL = ""

if PREVIEW_API_URL:
    probe = subprocess.run(
        [sys.executable, "-m", "evaluation.api_check", PREVIEW_API_URL],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    print(probe.stdout or probe.stderr)
else:
    print("Set PREVIEW_API_URL to run the deployed API check.")

Set PREVIEW_API_URL to run the deployed API check.


## 6. Review answer quality

Use the optional cell below to generate ten answers across direct questions, paraphrases, follow-ups, and out-of-corpus questions and inspect the evidence passed to the responder. The follow-up uses the conversation in the fixture. The out-of-corpus examples run with local evidence only so you can check whether the assistant acknowledges missing information.

Record a score of **0** for a miss, **1** for partial success, or **2** for full success on each dimension.

- **Correctness.** Check the facts and confirm that a follow-up answers the intended subject.
- **Citation support.** Check that each substantive claim is supported by the cited excerpt. For a valid abstention, award full credit when the answer avoids unsupported claims.
- **Coverage or abstention.** Check that the answer addresses the question's main parts or clearly identifies missing evidence.

Flag any score below 2 for correctness or citation support, and any zero for coverage, for review before publishing the demo. Report these human scores alongside retrieval metrics. An example score entry is `"exact-rag": (2, 1, 2)` for correctness, citation support, and coverage in that order.

In [8]:
REVIEW_IDS = (
    "exact-rag", "exact-transformer", "exact-metrics",
    "para-rag", "para-cross-topic", "para-grounding",
    "follow-rag", "follow-transformer",
    "out-geography", "out-same-domain",
)

if RUN_LIVE_ANSWER_REVIEW:
    import os
    from dataclasses import replace

    from dotenv import load_dotenv
    from langchain_core.messages import AIMessage, HumanMessage

    from graph import build_turn_graph
    from tools.research_providers import default_providers

    load_dotenv(ENV_FILE)
    if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
        raise RuntimeError("Set GOOGLE_API_KEY or GEMINI_API_KEY for live answer review.")

    providers = default_providers()
    review_graph = build_turn_graph(providers)
    local_graph = build_turn_graph(replace(providers, web_search=lambda _query: []))

    for case_id in REVIEW_IDS:
        case = cases_by_id[case_id]
        history = [
            (HumanMessage if index % 2 == 0 else AIMessage)(content=message)
            for index, message in enumerate(case.get("history", []))
        ]
        turn_graph = local_graph if case["group"] == "out_of_corpus" else review_graph
        result = turn_graph.invoke({"topic": case["topic"], "history": history})
        print(f"\n{case_id}  {case['topic']}")
        if history:
            print("Previous turns")
            for message in history:
                print(f"  {message.type}  {message.content}")
        print(f"Coverage  {result['coverage']}")
        print("Fallbacks", result.get("fallbacks", []))
        print(result["final_answer"])
        for number, source in enumerate(result["selected_candidates"], 1):
            print(f"[{number}] {source['url']}\n{source['content']}")
else:
    print("Set RUN_LIVE_ANSWER_REVIEW = True to collect answers and evidence for review.")

Set RUN_LIVE_ANSWER_REVIEW = True to collect answers and evidence for review.


In [9]:
# Fill this in after reviewing the answers above.
ratings = dict.fromkeys(REVIEW_IDS)
# Example after reviewing this answer
# ratings["exact-rag"] = (2, 1, 2)

for case_id, rating in ratings.items():
    if rating is not None:
        if len(rating) != 3 or any(score not in (0, 1, 2) for score in rating):
            raise ValueError(f"{case_id} needs three scores between 0 and 2.")

completed = [rating for rating in ratings.values() if rating is not None]
print(f"{len(completed)}/{len(ratings)} cases rated")
if completed:
    for label, scores in zip(("Correctness", "Citation support", "Coverage"), zip(*completed)):
        print(f"{label}  {sum(scores) / len(completed):.2f}/2")

flagged = [
    case_id for case_id, rating in ratings.items()
    if rating is not None and (rating[0] < 2 or rating[1] < 2 or rating[2] == 0)
]
if flagged:
    print("Cases to review", ", ".join(flagged))

0/10 cases rated


## 7. Record the result

Keep the run date, code revision, provider mode, scores for all three retrieval stages, out-of-corpus hits, mean and p95 latency, and gate status. Include any fallback categories and the number of answers you reviewed. This makes later runs comparable and keeps provider availability visible alongside ranking quality.

See the [README](README.md#verify-retrieval) for the CLI commands and the previous live run.